# MCP Review

> The course's review lesson is a video with no text. This is a faithful recap of what we built in this section (not a transcript).

**Model Context Protocol** relocates *who builds and maintains tools*: instead of hand-writing every integration, a **server** packages capabilities and any **client** connects to it. We built **both** sides (educational; in the real world you build one) as a CLI document chatbot.

## Architecture recap

```
MCP Client (your app)  ⇄  transport  ⇄  MCP Server(s)
                                          ├─ tools      (actions)
                                          ├─ resources  (data / GET-like)
                                          └─ prompts    (templated message sets)
```

- **Transports:** **stdio** (local subprocess — what our project uses) and **Streamable HTTP** (remote). Same messages either way.
- **Message types:** `ListTools`/`CallTool`, `ReadResource` (+ `ListResources`/`ListResourceTemplates`), `ListPrompts`/`GetPrompt`.
- **MCPClient** wraps the SDK's `ClientSession` and handles connect/cleanup (`AsyncExitStack`).

## The three primitives — server side vs client side

| Primitive | Purpose | Server (`mcp_server.py`) | Client (`mcp_client.py`) |
|-----------|---------|--------------------------|--------------------------|
| **Tools** | Perform actions | `@mcp.tool` → `read_doc_contents`, `edit_document` | `list_tools()`, `call_tool()` |
| **Resources** | Expose data | `@mcp.resource` → `docs://documents` (direct, JSON), `docs://documents/{doc_id}` (templated, text) | `read_resource(uri)` |
| **Prompts** | Pre-built instruction templates | `@mcp.prompt` → `format` (returns `base.Message` list) | `list_prompts()`, `get_prompt(name, args)` |

- **Tools** — Claude decides to call them; you execute and return results (the tool-use loop).
- **Resources** — *your app* fetches data by URI and injects it (no tool round-trip) → powers `@document` mentions.
- **Prompts** — server-authored, tested message templates → surfaced as `/slash-commands` (e.g. `/format`).
- The SDK auto-generates schemas from type hints + `Field`, and auto-serializes resource return values.

## What the finished project does

A CLI chatbot (`python main.py`) wired to a `DocumentMCP` server over in-memory docs:

- **Read/edit documents** via the two tools ("what's in report.pdf?", "change 20m to 35m").
- **`@`-mention** a document to inject its contents into the prompt (resources).
- **`/format`** a document into Markdown (a prompt that itself uses the `edit_document` tool).

All three primitives compose: a *prompt* starts the task, Claude uses a *tool* to act, and *resources* feed in context.

## Real-world takeaway

- You typically build **one** side: a **server** to expose your service, or you connect a **client** to existing servers.
- Existing clients register servers rather than reimplement them: **Claude Code** via `claude mcp add …`, the **Claude app** via Connectors, or your own client code.
- Write the server once → every MCP-speaking client can use it. (E.g. exposing a product's capabilities as an MCP server so Claude Code / claude.ai / your own UI all drive them.)

## This repo's adaptations (recap)

- **Two folders:** `cli-project/` (pristine starter — do the exercises) and `cli-project-complete/` (answer key). Starter's editable files are `skip-worktree` so local edits stay out of git.
- **Model:** `claude-haiku-4-5-20251001` (the code always sends `temperature=1.0`, which 400s on flagships).
- **Deps/runner:** pip mode (`USE_UV=0`) + shared `.venv` (`mcp[cli]`, `prompt-toolkit`), since `uv` isn't installed.
- **Import fix:** prompts use `from mcp.server.fastmcp.prompts import base` (the lesson's `from mcp.server.fastmcp import base` errors).
- **Inspector:** optional; newer versions need Node/`npx` and a proxy **session token** (or `DANGEROUSLY_OMIT_AUTH=true`).

## Optional exercise — the unimplemented prompt

`mcp_server.py` still carries `# TODO: Write a prompt to summarize a doc`. The course never implements it (even the complete server only ships `format`). It's a natural parallel to `format`: a `@mcp.prompt(name="summarize", ...)` taking a `doc_id`, returning a `base.UserMessage` that asks Claude to summarize the doc. A good self-check that you understand prompts.

Next: the **Quiz on Model Context Protocol** — the section's done.